# THUẬT TOÁN K-LÁNG GIỀNG GẦN NHẤT (K-NN)
**Tên:** Nguyễn Trung Kiên  
**Mssv:** 102230023  

### Bài tập:
1. **Số hóa & Khám phá Dữ liệu:** Khai thác tập dữ liệu 2D gồm **31 điểm huấn luyện** thuộc 4 lớp: $\mathcal{C} = \{\text{red triangle}, \text{blue square}, \text{green star}, \text{black heart}\}$ và **6 điểm kiểm thử chưa gán nhãn** ($A, B, C, D, E, F$).
2. **Câu a - Tính toán từng bước:** Lập bảng khoảng cách và phân lớp cho cả 6 điểm dưới **6 cấu hình** ($k \in \{4, 5, 6\}$ với khoảng cách Euclidean và Manhattan).
3. **Câu b - Báo cáo phân tích chuyên sâu:** Khảo sát tác động của kích thước láng giềng $k$, sự khác biệt hình học giữa Euclidean và Manhattan, cùng hiện tượng hòa phiếu (Tie-breaking).
4. **Câu c - Cài đặt KNN Scratch:** Viết chương trình Python thuần (không dùng scikit-learn) có hiển thị chi tiết bảng láng giềng, tỷ lệ phiếu bầu và chạy thực nghiệm minh họa cho điểm $E$ với $k = 5$.


In [1]:
import math
from collections import Counter
from typing import Dict, List, Tuple, Union

# =========================================================================
# 1. KHỞI TẠO TẬP DỮ LIỆU HUẤN LUYỆN TỪ HÌNH 1 (GRID 10x10)
# Classes: Red Triangle, Blue Square, Green Star, Black Heart
# =========================================================================

# Data sample format: ((x, y), class_label)
dataset_samples: List[Tuple[Tuple[float, float], str]] = [
    # Red Triangle (8 points)
    ((1, 9), 'Red Triangle'), ((3, 9), 'Red Triangle'), ((4, 7), 'Red Triangle'),
    ((2, 4), 'Red Triangle'), ((1, 2), 'Red Triangle'), ((3, 1), 'Red Triangle'),
    ((6, 5), 'Red Triangle'), ((6, 2), 'Red Triangle'),
    
    # Blue Square (9 points - includes (5, 3))
    ((1, 7), 'Blue Square'), ((3, 7), 'Blue Square'), ((7, 8), 'Blue Square'),
    ((9, 9), 'Blue Square'), ((8, 5), 'Blue Square'), ((5, 3), 'Blue Square'),
    ((8, 3), 'Blue Square'), ((9, 2), 'Blue Square'), ((8, 1), 'Blue Square'),
    
    # Green Star (5 points)
    ((4, 8), 'Green Star'), ((8, 7), 'Green Star'), ((4, 4), 'Green Star'),
    ((2, 1), 'Green Star'), ((8, 2), 'Green Star'),
    
    # Black Heart (9 points)
    ((2, 5), 'Black Heart'), ((3, 5), 'Black Heart'), ((4, 5), 'Black Heart'),
    ((1, 3), 'Black Heart'), ((3, 3), 'Black Heart'), ((2, 2), 'Black Heart'),
    ((4, 2), 'Black Heart'), ((1, 1), 'Black Heart'), ((5, 1), 'Black Heart')
]

# Unlabeled target points to predict
points_to_classify: Dict[str, Tuple[float, float]] = {
    'A': (2, 8),
    'B': (6, 7),
    'C': (7, 5),
    'D': (2, 3),
    'E': (7, 2),
    'F': (4, 1)
}

print(f"-> Total training samples loaded: {len(dataset_samples)} samples.")
print(f"-> Query points to test: {list(points_to_classify.keys())}")

-> Total training samples loaded: 31 samples.
-> Query points to test: ['A', 'B', 'C', 'D', 'E', 'F']


## Phần A: Bảng tính toán khoảng cách và kết quả phân loại từng bước

### 1. Cơ sở tính toán khoảng cách
Với điểm truy vấn $u = (x_u, y_u)$ và điểm huấn luyện $v = (x_v, y_v)$:
- **Khoảng cách Euclid (Chuẩn $L_2$):**  
  $$d_{\text{Euclid}}(u, v) = \sqrt{(x_u - x_v)^2 + (y_u - y_v)^2}$$
- **Khoảng cách Manhattan (Chuẩn $L_1$):**  
  $$d_{\text{Manhattan}}(u, v) = |x_u - x_v| + |y_u - y_v|$$

---

### 2. Bảng tổng hợp phân loại 6 điểm ($A \to F$) qua 6 trường hợp

| Điểm cần xét | Tọa độ | Độ đo khoảng cách | Kết quả với $k = 4$ | Kết quả với $k = 5$ | Kết quả với $k = 6$ |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Điểm A** | (2, 8) | **Euclidean** | [2 Sq, 2 Tr] $\rightarrow$ **Hòa phiếu (Square / Triangle)** | [2 Sq, 2 Tr, 1 St] $\rightarrow$ **Hòa phiếu (Square / Triangle)** | [3 Tr, 2 Sq, 1 St] $\rightarrow$ **Red Triangle** |
| | | **Manhattan** | [2 Sq, 2 Tr] $\rightarrow$ **Hòa phiếu (Square / Triangle)** | [2 Sq, 2 Tr, 1 St] $\rightarrow$ **Hòa phiếu (Square / Triangle)** | [3 Tr, 2 Sq, 1 St] $\rightarrow$ **Red Triangle** *(tùy tie-break)* |
| **Điểm B** | (6, 7) | **Euclidean** | [2 Tr, 1 Sq, 1 St] $\rightarrow$ **Red Triangle** | [2 Tr, 2 St, 1 Sq] $\rightarrow$ **Hòa phiếu (Triangle / Star)** | [2 Tr, 2 St, 2 Sq] $\rightarrow$ **Hòa phiếu (Tam mã)** |
| | | **Manhattan** | [2 Tr, 1 St, 1 Sq] $\rightarrow$ **Red Triangle** | [2 Tr, 2 Sq, 1 St] $\rightarrow$ **Hòa phiếu (Triangle / Square)** | [2 Tr, 2 St, 2 Sq] $\rightarrow$ **Hòa phiếu (Tam mã)** |
| **Điểm C** | (7, 5) | **Euclidean** | [2 Sq, 1 Tr, 1 St] $\rightarrow$ **Blue Square** | [3 Sq, 1 Tr, 1 St] $\rightarrow$ **Blue Square** | [3 Sq, 1 Tr, 1 St, 1 Ht] $\rightarrow$ **Blue Square** |
| | | **Manhattan** | [2 Sq, 1 Tr, 1 Ht] $\rightarrow$ **Blue Square** | [2 Sq, 1 Tr, 1 Ht, 1 St] $\rightarrow$ **Blue Square** | [3 Sq, 1 Tr, 1 Ht, 1 St] $\rightarrow$ **Blue Square** |
| **Điểm D** | (2, 3) | **Euclidean** | [3 Ht, 1 Tr] $\rightarrow$ **Black Heart** | [3 Ht, 2 Tr] $\rightarrow$ **Black Heart** | [3 Ht, 2 Tr, 1 St] $\rightarrow$ **Black Heart** |
| | | **Manhattan** | [3 Ht, 1 Tr] $\rightarrow$ **Black Heart** | [3 Ht, 1 Tr, 1 St] $\rightarrow$ **Black Heart** | [3 Ht, 2 Tr, 1 St] $\rightarrow$ **Black Heart** |
| **Điểm E** | (7, 2) | **Euclidean** | [2 Sq, 1 Tr, 1 St] $\rightarrow$ **Blue Square** | [3 Sq, 1 Tr, 1 St] $\rightarrow$ **Blue Square** | [3 Sq, 1 Tr, 1 St, 1 Ht] $\rightarrow$ **Blue Square** |
| | | **Manhattan** | [2 Sq, 1 Tr, 1 St] $\rightarrow$ **Blue Square** | [3 Sq, 1 Tr, 1 St] $\rightarrow$ **Blue Square** | [3 Sq, 1 Tr, 1 St, 1 Ht] $\rightarrow$ **Blue Square** |
| **Điểm F** | (4, 1) | **Euclidean** | [2 Ht, 1 Tr, 1 St] $\rightarrow$ **Black Heart** | [3 Ht hoặc 2 Ht, 2 Tr] $\rightarrow$ **Black Heart / Hòa** | [3 Ht hoặc 2 Ht, 2 Tr] $\rightarrow$ **Black Heart / Hòa** |
| | | **Manhattan** | [2 Ht, 1 Tr, 1 St] $\rightarrow$ **Black Heart** | [3 Ht hoặc 2 Ht, 2 Tr] $\rightarrow$ **Black Heart / Hòa** | [4 Ht hoặc 3 Ht, 2 Tr] $\rightarrow$ **Black Heart** |

*Ghi chú ký hiệu: Sq = Blue Square, Tr = Red Triangle, St = Green Star, Ht = Black Heart.*

## Phần B: Phân tích và đánh giá so sánh thực nghiệm

### 1. Đánh giá về việc lựa chọn số láng giềng K (K = 4, 5, 6)
- **Hiện tượng hòa phiếu khi K là số chẵn:**
  - Khi chọn số lượng hàng xóm là số chẵn ($K = 4$ hoặc $K = 6$), các lớp có xác suất rất cao bị trùng số lượng phiếu bầu. Điển hình như ở **điểm A** với $K = 4$, có chính xác 2 điểm Blue Square và 2 điểm Red Triangle cùng ở khoảng cách $\sqrt{2} \approx 1.414$, dẫn tới hòa phiếu 2 - 2. Khi đó nếu không có quy tắc bốc thăm phụ thì thuật toán không biết gán nhãn nào.
  - Do đó trong thực hành phân loại, nguyên tắc luôn là ưu tiên chọn **K lẻ** (như $K = 5$) để tránh rơi vào thế hòa.
- **Độ nhạy của các điểm ở vùng giáp ranh (như điểm B):**
  - Điểm B nằm ở ranh giới giao nhau giữa nhiều nhóm. Khi $K = 4$, lớp Red Triangle chiếm đa số. Tuy nhiên khi nới rộng bán kính lân cận lên $K = 5$, điểm B lấy thêm hàng xóm Green Star và rơi vào thế hòa 2 - 2. Điều này cho thấy việc tăng K có thể làm đảo chiều kết quả phân loại đối với các điểm nằm ở rìa.
- **Tính ổn định của các điểm nằm sâu trong cụm (như D, E, F):**
  - Xung quanh các điểm này có mật độ dữ liệu cùng loại rất dày đặc. Vì vậy dù có thay đổi $K = 4, 5$ hay $6$ thì nhãn dự đoán vẫn không hề lay chuyển (D và F luôn là Black Heart, E luôn là Blue Square với tỷ lệ phiếu áp đảo).

---

### 2. So sánh ảnh hưởng giữa khoảng cách Euclidean và Manhattan
- **Đặc trưng hình học:**
  - **Euclidean:** Tính khoảng cách trực tiếp theo đường chim bay ($d = \sqrt{\Delta x^2 + \Delta y^2}$). Lân cận tìm kiếm có dạng hình tròn, đối xử công bằng theo tất cả mọi hướng.
  - **Manhattan:** Tính khoảng cách di chuyển dọc theo trục lưới như bàn cờ ($d = |\Delta x| + |\Delta y|$). Lân cận tìm kiếm có dạng hình thoi, dẫn đến việc các điểm nằm trên đường chéo bị tính khoảng cách dài hơn thực tế (phạt nặng điểm chéo).
- **Tác động thực tế lên thứ hạng láng giềng:**
  - Ở **điểm A (2, 8)**, khi đo bằng Manhattan xuất hiện đến **5 điểm khác nhau cùng có khoảng cách bằng 2**. Lúc này việc chọn ra 4 hay 5 điểm láng giềng sẽ bị phụ thuộc vào thứ tự duyệt trong danh sách dữ liệu.
  - Tuy nhiên đối với các điểm nằm gọn trong cụm tập trung ($C, D, E, F$), cả hai phương pháp đo đều xác định được nhóm láng giềng chủ chốt giống nhau và trả về cùng một nhãn dự đoán.

## Phần C: Tự xây dựng thuật toán KNN From Scratch bằng Python

Thuật toán được hiện thực hóa theo đúng các yêu cầu kỹ thuật của đề bài:
* Không sử dụng thư viện mô hình có sẵn (`scikit-learn`).
* Sử dụng thuần các thư viện toán học cơ bản: `math`, `collections.Counter`.
* Cung cấp hàm dự đoán cho phép chọn điểm mục tiêu, giá trị $k$ và độ đo khoảng cách (`'euclidean'` hoặc `'manhattan'`).
* In đầy đủ các bước trung gian: bảng khoảng cách đã sắp xếp, $k$ hàng xóm gần nhất, thống kê tỷ lệ phiếu bầu và kết luận nhãn.

In [2]:
class CustomKNNClassifier:
    """
    Custom KNN Classifier built from scratch.
    Written by: Nguyen Trung Kien (102230023)
    """
    
    def __init__(self, k: int = 5, metric: str = 'euclidean') -> None:
        self.k = k
        self.metric = metric.lower()
        self.data_samples: List[Tuple[Tuple[float, float], str]] = []
        
    def fit(self, samples: List[Tuple[Tuple[float, float], str]]) -> None:
        """Store training samples."""
        self.data_samples = samples
        
    @staticmethod
    def compute_distance(p1: Tuple[float, float], p2: Tuple[float, float], metric: str) -> float:
        """Calculate metric distance between two coordinate pairs."""
        dx = p1[0] - p2[0]
        dy = p1[1] - p2[1]
        if metric == 'euclidean':
            return math.sqrt(dx * dx + dy * dy)
        elif metric == 'manhattan':
            return abs(dx) + abs(dy)
        else:
            raise ValueError(f"Unknown distance metric: {metric}")
            
    def predict_target(self, target_id: str, verbose: bool = True) -> Tuple[str, Dict[str, int]]:
        """
        Predict label for a specified query point with formatted step-by-step logs.
        """
        if target_id not in points_to_classify:
            raise KeyError(f"Target point '{target_id}' is not defined.")
            
        query_coord = points_to_classify[target_id]
        
        # Step 1: Calculate distance to all training samples
        neighbor_records = []
        for coord, label in self.data_samples:
            dist = self.compute_distance(query_coord, coord, self.metric)
            neighbor_records.append({
                'coord': coord,
                'label': label,
                'dist': dist
            })
            
        # Step 2: Sort neighbors by distance ascending
        neighbor_records.sort(key=lambda item: item['dist'])
        
        # Step 3: Extract top K nearest neighbors
        nearest_k = neighbor_records[:self.k]
        
        # Step 4: Count votes for each class
        vote_counter = Counter([item['label'] for item in nearest_k])
        highest_votes = vote_counter.most_common(1)[0][1]
        top_classes = [c for c, v in vote_counter.items() if v == highest_votes]
        result_label = " / ".join(top_classes)
        
        if verbose:
            print(f"\n{'#'*65}")
            print(f"# EXPERIMENT KNN - POINT {target_id} {query_coord} | K = {self.k} | METRIC: {self.metric.upper()}")
            print(f"{'#'*65}")
            
            print("\n[1] Top 8 Nearest Neighbors (Sorted):")
            print(f"{'Rank':<10} | {'Coord':<10} | {'Label':<15} | {'Distance':<10}")
            print("-" * 55)
            for idx, item in enumerate(neighbor_records[:8], start=1):
                pt_str = f"({item['coord'][0]}, {item['coord'][1]})"
                print(f"{idx:<10} | {pt_str:<10} | {item['label']:<15} | {item['dist']:.4f}")
                
            print(f"\n[2] Selected {self.k} Nearest Neighbors:")
            for idx, item in enumerate(nearest_k, start=1):
                pt_str = f"({item['coord'][0]}, {item['coord'][1]})"
                print(f"  + Neighbor #{idx}: {pt_str:<10} | Class: {item['label']:<15} | d = {item['dist']:.4f}")
                
            print("\n[3] Voting Summary:")
            for cls_name, votes in vote_counter.items():
                ratio = (votes / self.k) * 100
                print(f"  - Class '{cls_name}': {votes} votes ({ratio:.1f}%)")
                
            print("\n[4] Final Decision:")
            if len(top_classes) > 1:
                print(f"  => RESULT: TIE DETECTED between {result_label}")
            else:
                print(f"  => RESULT: ASSIGNED CLASS -> [{result_label.upper()}]")
            print(f"{'#'*65}\n")
            
        return result_label, dict(vote_counter)


# Instantiate model
knn_model = CustomKNNClassifier()
knn_model.fit(dataset_samples)

### Thực nghiệm kiểm thử (Demonstration)
Chạy mô hình dự đoán cho **điểm E** với cấu hình **$k = 5$** sử dụng lần lượt cả hai độ đo **Euclidean** và **Manhattan**.

In [3]:
# Demo 1: Point E with Euclidean metric
knn_model.k = 5
knn_model.metric = 'euclidean'
knn_model.predict_target('E', verbose=True)

# Demo 2: Point E with Manhattan metric
knn_model.metric = 'manhattan'
knn_model.predict_target('E', verbose=True)


#################################################################
# EXPERIMENT KNN - POINT E (7, 2) | K = 5 | METRIC: EUCLIDEAN
#################################################################

[1] Top 8 Nearest Neighbors (Sorted):
Rank       | Coord      | Label           | Distance  
-------------------------------------------------------
1          | (6, 2)     | Red Triangle    | 1.0000
2          | (8, 2)     | Green Star      | 1.0000
3          | (8, 3)     | Blue Square     | 1.4142
4          | (8, 1)     | Blue Square     | 1.4142
5          | (9, 2)     | Blue Square     | 2.0000
6          | (5, 3)     | Blue Square     | 2.2361
7          | (5, 1)     | Black Heart     | 2.2361
8          | (4, 2)     | Black Heart     | 3.0000

[2] Selected 5 Nearest Neighbors:
  + Neighbor #1: (6, 2)     | Class: Red Triangle    | d = 1.0000
  + Neighbor #2: (8, 2)     | Class: Green Star      | d = 1.0000
  + Neighbor #3: (8, 3)     | Class: Blue Square     | d = 1.4142
  + Neighbor #

('Blue Square', {'Red Triangle': 1, 'Green Star': 1, 'Blue Square': 3})

### Bảng kiểm chứng tự động toàn bộ 6 điểm ($A \to F$)
Khối lệnh dưới đây tự động chạy qua tất cả 6 điểm truy vấn và 6 thiết lập để đối chiếu nhanh với bảng lý thuyết ở Phần A.

In [4]:
print(f"{'Point':<6} | {'Metric':<10} | {'k = 4':<25} | {'k = 5':<25} | {'k = 6':<25}")
print("=" * 85)

for pt_name in ['A', 'B', 'C', 'D', 'E', 'F']:
    for m in ['euclidean', 'manhattan']:
        res = []
        for k_val in [4, 5, 6]:
            knn_model.k = k_val
            knn_model.metric = m
            predicted, _ = knn_model.predict_target(pt_name, verbose=False)
            res.append(predicted)
        print(f"{pt_name:<6} | {m.capitalize():<10} | {res[0]:<25} | {res[1]:<25} | {res[2]:<25}")

Point  | Metric     | k = 4                     | k = 5                     | k = 6                    
A      | Euclidean  | Red Triangle / Blue Square | Red Triangle / Blue Square | Red Triangle             
A      | Manhattan  | Red Triangle / Blue Square | Red Triangle / Blue Square | Red Triangle             
B      | Euclidean  | Red Triangle              | Red Triangle / Green Star | Blue Square / Red Triangle / Green Star
B      | Manhattan  | Red Triangle              | Red Triangle / Blue Square | Red Triangle / Blue Square / Green Star
C      | Euclidean  | Blue Square               | Blue Square               | Blue Square              
C      | Manhattan  | Blue Square               | Blue Square               | Blue Square              
D      | Euclidean  | Black Heart               | Black Heart               | Black Heart              
D      | Manhattan  | Black Heart               | Black Heart               | Black Heart              
E      | Euclidean  | Blue Squa